# 13 - Word2Vec from Scratch with HuggingFace Dataset

Goal: Train Word2Vec on Spanish corpus from HuggingFace

Run with: conda activate tfenv

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Load Spanish corpus using HuggingFace datasets API
print("Loading Spanish corpus from HuggingFace...")
from datasets import load_dataset

# Load alpindale/light-novels dataset
ds = load_dataset("alpindale/light-novels", split="train")

# Filter for Spanish texts
print("Filtering Spanish texts...")
spanish_texts = [row['text'] for row in ds if row.get('language', '') == 'es']

if len(spanish_texts) == 0:
    # If no language filter, just take first texts
    spanish_texts = [row['text'] for row in ds][:20000]

print(f"Loaded {len(spanish_texts)} Spanish texts")

In [ ]:
# Combine texts
full_text = ' '.join(spanish_texts[:10000])  # Use first 10k texts
print(f"Total words: {len(full_text.split())}")

In [ ]:
# Build vocabulary
words = full_text.lower().split()
words = [w.strip('.,;:!?()[]"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2]

from collections import Counter
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c >= 5]

text_vocab = {word: idx for idx, word in enumerate(vocab)}
text_vocab_size = len(text_vocab)
text_embed_dim = 64

print(f"Vocabulary: {text_vocab_size} words")

In [ ]:
# Create training pairs (Skip-gram)
def create_pairs(words, window=2):
    pairs = []
    for i, word in enumerate(words):
        if word not in text_vocab:
            continue
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j and words[j] in text_vocab:
                pairs.append((text_vocab[word], text_vocab[words[j]]))
    return pairs

pairs = create_pairs(words)
train_words = np.array([p[0] for p in pairs], dtype=np.int32)
train_context = np.array([p[1] for p in pairs], dtype=np.int32)
print(f"Training pairs: {len(pairs)}")

In [ ]:
# Word2Vec from scratch using Keras
embedding_layer = layers.Embedding(
    input_dim=text_vocab_size,
    output_dim=text_embed_dim,
    name='embedding'
)

model = keras.Sequential([
    embedding_layer,
    layers.Flatten(),
    layers.Dense(text_vocab_size, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.02),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Word2Vec model ready!")
model.summary()

In [ ]:
# Efficient training with tf.data
train_ds = tf.data.Dataset.from_tensor_slices((train_words, train_context))
train_ds = train_ds.shuffle(buffer_size=len(train_words)).batch(512).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec...")
start = time.time()

history = model.fit(
    train_ds,
    epochs=10,
    verbose=1
)

print(f"Training time: {time.time() - start:.2f}s")

In [ ]:
# Get embeddings
embeddings = embedding_layer.get_weights()[0]
print(f"Embeddings shape: {embeddings.shape}")

# Find similar words
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words ===")
test_words = ['tecnología', 'datos', 'desarrollo', 'empresa', 'aprendizaje', 'vida']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

In [ ]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word Embeddings (PCA 3D) - Top 200 words')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.show()